In [1]:
import torch

## NN have 2 main things going on feature extraction + Classification/Regression


before AlexNet human would have manually extract features and then these feats were passed to a classification model like rf, gboost. 

#### But how do we know which feat is significant. 

what nn do is they extract both at exact same time. 



In [1]:
import torch ; import os
import torch.nn as nn
import torch.optim as optim
from torchvision.models import AlexNet, AlexNet_Weights
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from tqdm import tqdm
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
PATH_TO_DATA = 'datasets/PetImages'
os.listdir(PATH_TO_DATA)

['Cat', 'Dog']

In [3]:
from PIL import Image
import os

bad_files = []

root = "datasets/PetImages"

for cls in ["Cat", "Dog"]:
    folder = os.path.join(root, cls)

    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)

        try:
            with Image.open(path) as img:
                img.verify()
        except Exception:
            bad_files.append(path)

print(f"Found {len(bad_files)} corrupted files")

for f in bad_files:
    print(f)

Found 0 corrupted files


In [4]:
for f in bad_files:
    os.remove(f)

In [5]:
# zscore standardizing data, but why?
normalizer = transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])



train_transforms = transforms.Compose(
    [
        transforms.Resize((224,224)), # bcos alexnet expects this size
        transforms.RandomHorizontalFlip(p=0.5), # data augmentation
        transforms.ToTensor(),
        normalizer
    ]
)
train_transforms
dataset = ImageFolder(PATH_TO_DATA, transform = train_transforms)

train_samples, test_samples = int(0.9*len(dataset)), len(dataset) - int(0.9*len(dataset))


train_samples, test_samples

(22498, 2500)

In [6]:
train_dataset , val_dataset = torch.utils.data.random_split(dataset, lengths=[train_samples, test_samples])

In [7]:
for sample in dataset: 
    print(sample)
    break

(tensor([[[ 1.9749,  2.0092,  2.0605,  ...,  1.4440,  1.3927,  1.3584],
         [ 1.9920,  2.0263,  2.0605,  ...,  1.4440,  1.3927,  1.3584],
         [ 1.9920,  2.0263,  2.0434,  ...,  1.4440,  1.3927,  1.3584],
         ...,
         [-2.0665, -2.0665, -2.0665,  ...,  0.5536,  0.5364,  0.5193],
         [-2.0837, -2.0837, -2.0837,  ...,  0.5193,  0.5022,  0.5022],
         [-2.1008, -2.1008, -2.1008,  ...,  0.5022,  0.4851,  0.4679]],

        [[ 1.4657,  1.5007,  1.5182,  ...,  0.9230,  0.8704,  0.8354],
         [ 1.4832,  1.5007,  1.5357,  ...,  0.9230,  0.8704,  0.8354],
         [ 1.4832,  1.5182,  1.5532,  ...,  0.9230,  0.8704,  0.8354],
         ...,
         [-1.9832, -1.9832, -1.9832,  ...,  0.1527,  0.1352,  0.1176],
         [-2.0007, -2.0007, -2.0007,  ...,  0.1176,  0.1001,  0.1001],
         [-2.0182, -2.0182, -2.0182,  ...,  0.1001,  0.0826,  0.0651]],

        [[ 0.3045,  0.3219,  0.3045,  ..., -0.2010, -0.2532, -0.2881],
         [ 0.3219,  0.3219,  0.3219,  ..., -

In [8]:
model = AlexNet() # AlexNet isn't a pretrained model, it is a randomly initialized model ready to be pretrained
model # meaning the weights are randomly initialized. 

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [9]:
model.features # accessing the feat part

Sequential(
  (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
  (1): ReLU(inplace=True)
  (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (4): ReLU(inplace=True)
  (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU(inplace=True)
  (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (9): ReLU(inplace=True)
  (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
)

In [10]:
model.features[1] , model.classifier[1]

(ReLU(inplace=True), Linear(in_features=9216, out_features=4096, bias=True))

In [11]:
model.classifier[6]

Linear(in_features=4096, out_features=1000, bias=True)

In [12]:
model.classifier[6] = nn.Linear(4096, 2) # changing the layer to match our dogvscat dataset.
# 4096 inputs bcos the previous layer  has 4096 out features
model.classifier

Sequential(
  (0): Dropout(p=0.5, inplace=False)
  (1): Linear(in_features=9216, out_features=4096, bias=True)
  (2): ReLU(inplace=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=4096, out_features=4096, bias=True)
  (5): ReLU(inplace=True)
  (6): Linear(in_features=4096, out_features=2, bias=True)
)

In [13]:
rand_data = torch.randn(16,3, 224, 224) # random 16 images with rgb channels and 224x224 pixels

In [14]:
model(rand_data).shape # for the 15 of them we have 2 output for each. 

torch.Size([16, 2])

In [15]:
# we care how many learnable models are there in the model 
for name, param in model.named_parameters():
    print(name) # these are all the learnable params in model

features.0.weight
features.0.bias
features.3.weight
features.3.bias
features.6.weight
features.6.bias
features.8.weight
features.8.bias
features.10.weight
features.10.bias
classifier.1.weight
classifier.1.bias
classifier.4.weight
classifier.4.bias
classifier.6.weight
classifier.6.bias


In [16]:
num_params = 0

for name, paramn in model.named_parameters():
    print(param)
    break

Parameter containing:
tensor([-0.0119, -0.0116], requires_grad=True)


In [17]:
for name, param in model.named_parameters():
    print(
        f"{name:35}",
        tuple(param.shape),
        param.numel()
    )

features.0.weight                   (64, 3, 11, 11) 23232
features.0.bias                     (64,) 64
features.3.weight                   (192, 64, 5, 5) 307200
features.3.bias                     (192,) 192
features.6.weight                   (384, 192, 3, 3) 663552
features.6.bias                     (384,) 384
features.8.weight                   (256, 384, 3, 3) 884736
features.8.bias                     (256,) 256
features.10.weight                  (256, 256, 3, 3) 589824
features.10.bias                    (256,) 256
classifier.1.weight                 (4096, 9216) 37748736
classifier.1.bias                   (4096,) 4096
classifier.4.weight                 (4096, 4096) 16777216
classifier.4.bias                   (4096,) 4096
classifier.6.weight                 (2, 4096) 8192
classifier.6.bias                   (2,) 2


In [18]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
!nvidia-smi

Mon Jul 13 23:46:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5050 ...    Off |   00000000:04:00.0  On |                  N/A |
| N/A   42C    P8             13W /  100W |     882MiB /   8151MiB |      5%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
DEVICE

'cuda'

In [31]:
model = AlexNet()
model.classifier[6] = nn.Linear(4096, 2)
model = model.to(DEVICE)

epochs = 5
optimizer = optim.Adam(model.parameters(), lr = 0.0001)
loss_fn = nn.CrossEntropyLoss()
batch_size = 128


In [21]:
trainloader = DataLoader(train_dataset, batch_size = batch_size , shuffle = True, num_workers = 4)
valloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False, num_workers = 4)

In [29]:
from tqdm.auto import tqdm
import torch

def train(model, device, epochs, optimizer, loss_fn, trainloader, valloader):

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    model.to(device)

    for epoch in range(epochs):

        
        # TRAINING
        
        model.train()

        running_loss = 0.0
        running_correct = 0
        total = 0

        pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}")

        for images, labels in pbar:

            images = images.to(device)
            labels = labels.to(device)

            # Forward
            outputs = model(images)
            loss = loss_fn(outputs, labels)

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Statistics
            running_loss += loss.item()

            preds = outputs.argmax(dim=1)
            running_correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix(
                loss=running_loss/(total/images.size(0)),
                acc=100*running_correct/total
            )

        train_loss = running_loss / len(trainloader)
        train_acc = running_correct / total

        
        # VALIDATION
        
        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in valloader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = loss_fn(outputs, labels)

                val_loss += loss.item()

                preds = outputs.argmax(dim=1)

                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= len(valloader)
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"\nEpoch {epoch+1}/{epochs}"
            f"\nTrain Loss : {train_loss:.4f}"
            f"\nTrain Acc  : {train_acc:.4f}"
            f"\nVal Loss   : {val_loss:.4f}"
            f"\nVal Acc    : {val_acc:.4f}"
        )

    return history, model

In [23]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0
13.0
True
NVIDIA GeForce RTX 5050 Laptop GPU


In [26]:
history, model = train(
    model=model,
    device=DEVICE,
    epochs=epochs,
    optimizer=optimizer,
    loss_fn=loss_fn,
    trainloader=trainloader,
    valloader=valloader
)

Epoch 1/5: 100%|████████████████████████████| 176/176 [00:19<00:00,  9.25it/s, acc=63.9, loss=0.48]



Epoch 1/5
Train Loss : 0.6267
Train Acc  : 0.6392
Val Loss   : 0.5616
Val Acc    : 0.7292


Epoch 2/5: 100%|█████████████████████████████| 176/176 [00:19<00:00,  9.10it/s, acc=76, loss=0.379]



Epoch 2/5
Train Loss : 0.4938
Train Acc  : 0.7602
Val Loss   : 0.4644
Val Acc    : 0.7840


Epoch 3/5: 100%|████████████████████████████| 176/176 [00:19<00:00,  8.92it/s, acc=81.8, loss=0.31]



Epoch 3/5
Train Loss : 0.4047
Train Acc  : 0.8178
Val Loss   : 0.3991
Val Acc    : 0.8188


Epoch 4/5: 100%|███████████████████████████| 176/176 [00:19<00:00,  8.89it/s, acc=84.8, loss=0.264]



Epoch 4/5
Train Loss : 0.3440
Train Acc  : 0.8485
Val Loss   : 0.3121
Val Acc    : 0.8608


Epoch 5/5: 100%|███████████████████████████| 176/176 [00:19<00:00,  9.11it/s, acc=87.1, loss=0.232]



Epoch 5/5
Train Loss : 0.3021
Train Acc  : 0.8707
Val Loss   : 0.2905
Val Acc    : 0.8728


Lets Now Load our PreTrained Weights

We will still be training the entire model end to end, but it will now start with the pretrained weights from ImageNet.

Note: The pretrained model by default has a final linear layer that outputs to 1000 classes! When we swap this last portion out with a new Linear layer that outputs to 2 classes, only that layer will be randomly initialized. The remaining calculations (previous linear layers, convolutions, etc...) are all still using the pretrained valued.

In [33]:
model = torch.hub.load('pytorch/vision:v0.10.0', 'alexnet', pretrained=True)
model.classifier[6] = nn.Linear(4096, 2)
model = model.to(DEVICE)

### MODEL TRAINING INPUTS ###
epochs = 2
optimizer = optim.Adam(params=model.parameters(), lr=0.0001)
loss_fn = nn.CrossEntropyLoss()
batch_size = 128

### BUILD DATALOADERS ###
trainloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
valloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)


random_init_logs, model = train(model=model,
                                device=DEVICE,
                                epochs=epochs,
                                optimizer=optimizer,
                                loss_fn=loss_fn,
                                trainloader=trainloader,
                                valloader=valloader)

Using cache found in /home/blixture/.cache/torch/hub/pytorch_vision_v0.10.0
Epoch 1/2: 100%|████████████████████████████| 176/176 [00:19<00:00,  8.99it/s, acc=95, loss=0.0935]



Epoch 1/2
Train Loss : 0.1219
Train Acc  : 0.9496
Val Loss   : 0.1078
Val Acc    : 0.9568


Epoch 2/2: 100%|██████████████████████████| 176/176 [00:19<00:00,  8.99it/s, acc=97.1, loss=0.0553]



Epoch 2/2
Train Loss : 0.0721
Train Acc  : 0.9713
Val Loss   : 0.0942
Val Acc    : 0.9644


Notice in a less number of epochs, we have beat the preformance of the "From Scratch" model greatly. Obviously if we kept training the randomized model more, we would have reached a similar performance but I am trying to show the benefits if you had lower resources for compute.
Load PreTrained Weights but Only Train the Final Classifier Layer

Like we had mentioned previously, the entire model starts with the pretrained weights, but when we swap the last linear layer with one that outputs to 2 classes, that layer will become randomly initialized. A common technique is then to freeze the remaining layers and only train this classifier head. To do this lets look at a flag that many tensors have in PyTorch!

Note: All Tensors (atleast by default) in your PyTorch model will have this. You will see that I am only showing the first layer that includes "bias" in the name. This is only for visualization as the bias tensors are pretty small compared to the massive weight tensors that belong to the convolutions. The idea is the same regardless though as the bias is also a learnable parameter!

In [34]:
for name, params in model.named_parameters():
    if 'bias' in name:
        print(name)
        print(params)
        break

features.0.bias
Parameter containing:
tensor([-0.9695, -2.8071, -0.0373, -0.0787, -0.1167,  0.0255, -0.0762, -1.4176,
         1.6436, -0.1004, -0.0171, -0.1287, -0.0669, -0.0340, -0.0745, -1.2978,
        -0.0494,  0.0116, -0.1023, -1.1879, -0.1378, -0.0488, -0.0785, -0.0418,
        -0.0967, -0.0713, -1.9368, -0.0874, -0.1395, -0.1974, -0.1283, -2.0089,
        -0.0503, -0.0634, -0.0370, -0.3841, -2.7823,  0.6605, -0.1653, -2.1298,
         0.0534, -0.0282, -0.1700, -0.0610, -0.4214, -1.9399, -1.2086,  0.0137,
        -0.1085, -0.0254, -0.1515, -1.8520, -0.0930, -0.0175, -0.0702, -0.0581,
        -0.0624, -0.0730, -1.2680, -0.1170, -0.0449, -0.3271,  0.0495, -0.0153],
       device='cuda:0', requires_grad=True)


Notice the flag at the end of the Tensor:

required_grad = True

This indicates that when PyTorch is optimizing the model, this tensor allows for gradient updates. Therefore, if we want to turn this off for this layer, we just need to turn that flag to False. We will want to repeat this process of turning off gradient updates for all layers except for our last one!

In [35]:
for name, param in model.named_parameters():
    print(name)

features.0.weight
features.0.bias
features.3.weight
features.3.bias
features.6.weight
features.6.bias
features.8.weight
features.8.bias
features.10.weight
features.10.bias
classifier.1.weight
classifier.1.bias
classifier.4.weight
classifier.4.bias
classifier.6.weight
classifier.6.bias


In [36]:
model = torch.hub.load('pytorch/vision:v0.10.0', 'alexnet', pretrained=True)
model.classifier[6] = nn.Linear(4096, 2)

# Check the name of all the parameters
for name, param in model.named_parameters():
    if "classifier.6" not in name:
        param.requires_grad_(False) # Inplace turn of gradient updates

        
for name, param in model.named_parameters():
    if "bias" in name:
        print(name)
        print(param)
        break

Using cache found in /home/blixture/.cache/torch/hub/pytorch_vision_v0.10.0


features.0.bias
Parameter containing:
tensor([-0.9705, -2.8070, -0.0371, -0.0795, -0.1159,  0.0252, -0.0752, -1.4181,
         1.6454, -0.0990, -0.0161, -0.1282, -0.0658, -0.0345, -0.0743, -1.2977,
        -0.0505,  0.0121, -0.1013, -1.1887, -0.1380, -0.0492, -0.0789, -0.0405,
        -0.0958, -0.0705, -1.9374, -0.0850, -0.1388, -0.1968, -0.1279, -2.0095,
        -0.0476, -0.0604, -0.0351, -0.3843, -2.7823,  0.6605, -0.1655, -2.1293,
         0.0543, -0.0274, -0.1703, -0.0593, -0.4215, -1.9394, -1.2094,  0.0153,
        -0.1081, -0.0248, -0.1503, -1.8516, -0.0928, -0.0177, -0.0700, -0.0582,
        -0.0630, -0.0721, -1.2678, -0.1176, -0.0441, -0.3259,  0.0507, -0.0146])


Notice now that the Requires Grad flag is now gone! We should be good to go now to retrain the model one more time.

In [38]:
model = torch.hub.load('pytorch/vision:v0.10.0', 'alexnet', pretrained=True)
model.classifier[6] = nn.Linear(4096, 2)

# Check the name of all the parameters
for name, param in model.named_parameters():
    if "classifier.6" not in name:
        param.requires_grad_(False) # Inplace turn of gradient updates

model = model.to(DEVICE)

### MODEL TRAINING INPUTS ###
epochs = 2
optimizer = optim.Adam(params=model.parameters(), lr=0.0001)
loss_fn = nn.CrossEntropyLoss()
batch_size = 128

### BUILD DATALOADERS ###
trainloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
valloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)


random_init_logs, model = train(model=model,
                                device=DEVICE,
                                epochs=epochs,
                                optimizer=optimizer,
                                loss_fn=loss_fn,
                                trainloader=trainloader,
                                valloader=valloader)

Using cache found in /home/blixture/.cache/torch/hub/pytorch_vision_v0.10.0
Epoch 1/2: 100%|███████████████████████████| 176/176 [00:14<00:00, 12.54it/s, acc=90.3, loss=0.176]



Epoch 1/2
Train Loss : 0.2300
Train Acc  : 0.9034
Val Loss   : 0.1353
Val Acc    : 0.9552


Epoch 2/2: 100%|██████████████████████████| 176/176 [00:14<00:00, 12.48it/s, acc=94.9, loss=0.0994]



Epoch 2/2
Train Loss : 0.1297
Train Acc  : 0.9488
Val Loss   : 0.1140
Val Acc    : 0.9584


## CONCLUSION:
Pretrained models allow us to "Transfer" knowledge from a previous task to a new one. The main benefits to this are as follows:

    Less data needed to train model and allows us to work on niche tasks even with these expressive datasets
    Less concern for Overfitting. Typically, if you train a large model on a small dataset, it will quickly memorize the data. Instead, we can train just a small part of the model and keep the rest as a feature extractor.
    Less compute! Deep Learning takes long enough already to train. Atleast we have less parameters to do gradient updating on, so it saves us quite a bit of compute.

Pretrained models are the current future for deep learning. It has become the gold standard to take large language, vision, speech, and any other modality and transfer it to new tasks. Even more interesting is, the models have become so powerful, they can often transfer to completely unrelated tasks (Using NLP based models for Protein Analysis). This technique also greatly democratizes Deep Learning to many because if the models are easier to train and dont require entire data centers like the Pretraining stage did, then we can all enjoy solving the problems we are interested in!